# LLM-as-a-judge (Claude) with human-labeled examples

Each transcript is classified by Claude in **its own API request** (no memory between classifications): your prompt, with labeled examples inserted, followed by the transcript; the answer is constrained to `{"level": "Low" | "Medium" | "High"}` by structured outputs.

**Same protocol as `human_fgw.ipynb` and `human_embeddings.ipynb`**, so the three methods are directly comparable:
* the same stratified folds and the **same examples** (read from `splits/`, created by whichever notebook runs first);
* for each fold, the examples ($m$ per level, from the other folds) are labeled either by the **humans** or by the **reference** level, and the transcripts of the fold are classified with them;
* **zero-shot**: the prompt without any example (the counterpart of the unsupervised setting);
* every classification is scored against both the human and the reference level (accuracy, macro-F1, quadratic-weighted $\kappa$ with 95% bootstrap CI), with the same section per generator model.

**Cost**: one request per transcript and per mode. Nothing is sent while `RUN_API = False`: the notebook prints the number of requests and a rough cost estimate first. Every answer is saved in `results/`, so a rerun never pays twice for the same request. The part of the prompt before the transcript (instructions + examples) is identical within a fold and is cached by the API.

In [ ]:
import hashlib
import json
import os
import re
import sys
from concurrent.futures import ThreadPoolExecutor

import anthropic
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, evaluate, kappa_with_ci, load_labeled_transcripts, load_or_create_examples,
                        load_or_create_split, plot_confusion)

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
REFERENCE_COLUMN = None         # reference (instructed) level; None: a column whose name contains "ref" or "instruct"
GENERATOR_COLUMN = None         # LLM that generated each video; None: a column whose name contains "model", "generator" or "llm"
DIMENSION = "ability"           # names the splits and outputs

SPLIT_DIR = "splits"            # shared with human_fgw.ipynb and human_embeddings.ipynb
OUTPUT_DIR = "results"

M = 10                          # examples per level (None: all the transcripts of the other folds)
N_FOLDS = 5
STRATIFY_ON = "reference"       # same as in the other notebooks
SEED = 0
N_REPEATS = 1                   # random splits (each one costs a full new set of requests)

# Your prompt, in a text file: write {examples} where the examples go and {transcript} where the
# transcript to classify goes (literal replacement: the braces of the tags are not a problem).
PROMPT_FILE = "llm_prompt.txt"
SYSTEM_PROMPT = None            # optional system prompt
EXAMPLE_TEMPLATE = "Transcript: {text}\nLevel: {level}"   # how each example is written
EXAMPLE_SEPARATOR = "\n\n"
NO_EXAMPLES_TEXT = ""           # what replaces {examples} in the zero-shot prompt

MODEL = "claude-opus-5-5"
EFFORT = "medium"               # low | medium | high | xhigh | max (thinking depth; "medium" is this model's default)
MAX_TOKENS = 8000               # includes the model's thinking
USE_FALLBACK = True             # if Claude declines a request, the API re-runs it on Anthropic's recommended fallback model
MAX_WORKERS = 8                 # parallel requests
RUN_API = False                 # set to True to send the requests (check the cost estimate first)

# $ per million tokens, for the cost estimate (claude-opus-5-5; update if you change MODEL)
PRICES = {"input": 4.0, "cache_write": 5.0, "cache_read": 0.2, "output": 20.0}
OUTPUT_TOKENS_GUESS = 400       # per request, thinking included (rough)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Labeled transcripts

In [ ]:
if REFERENCE_COLUMN is None:
    columns = pd.read_csv(LABELS_FILE, nrows=0).columns
    REFERENCE_COLUMN = next((c for c in columns if c not in (TEXT_COLUMN, LABEL_COLUMN)
                             and re.search("ref|instruct", c, re.IGNORECASE)), None)
print("reference column:", REFERENCE_COLUMN)

df = load_labeled_transcripts(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN, REFERENCE_COLUMN)
LABELS = {"human": "level", **({"reference": "reference"} if REFERENCE_COLUMN else {})}  # label -> column of df
texts = dict(zip(df["id"], df["text"]))
print(f"{len(df)} transcripts")
pd.DataFrame({label: df[column].value_counts() for label, column in LABELS.items()}).reindex(LEVELS).fillna(0).astype(int)

## Prompt

The examples of a fold are written with `EXAMPLE_TEMPLATE`, in a random order (fixed by the seed and the fold) so that the levels are not presented in a systematic order.

In [ ]:
with open(PROMPT_FILE, encoding="utf-8") as f:
    TEMPLATE = f.read()
assert "{transcript}" in TEMPLATE, "the prompt must contain {transcript}"
if "{examples}" not in TEMPLATE:
    print("Warning: the prompt has no {examples}: the few-shot modes will be identical to zero-shot.")


def format_examples(examples, seed):
    # Examples (DataFrame with id and level) written with EXAMPLE_TEMPLATE, in a seeded random order.
    if examples is None:
        return NO_EXAMPLES_TEXT
    order = np.random.default_rng(seed).permutation(len(examples))
    rows = examples.iloc[order]
    return EXAMPLE_SEPARATOR.join(EXAMPLE_TEMPLATE.replace("{text}", texts[i]).replace("{level}", level)
                                  for i, level in zip(rows["id"], rows["level"]))


def build_prompt(examples_text, transcript):
    # The prompt split in two: the part before the transcript (cached) and the rest.
    before, after = TEMPLATE.replace("{examples}", examples_text).split("{transcript}", 1)
    return before, transcript + after


STRATIFY = STRATIFY_ON if STRATIFY_ON in LABELS else "human"
print("folds stratified on the", STRATIFY, "level")
folds0, _ = load_or_create_split(df.assign(level=df[LABELS[STRATIFY]]), DIMENSION, f"strat{STRATIFY}_seed{SEED}",
                                 SPLIT_DIR, M, N_FOLDS, SEED)
examples0 = load_or_create_examples(folds0, df.set_index("id").loc[folds0["id"], "level"],
                                    f"{DIMENSION}_strat{STRATIFY}_human_seed{SEED}", SPLIT_DIR, M, SEED)
before, after = build_prompt(format_examples(examples0[examples0["fold"] == 0], SEED), texts[df["id"].iloc[0]])
print(f"Example of a full prompt (fold 0, human examples), {len(before) + len(after)} characters:\n")
print(before[:1500] + ("\n[...]\n" if len(before) > 3000 else "") + before[-1500:] + after)

## Requests

One request per transcript and per mode. Answers are saved in `results/llm_answers_<DIMENSION>.jsonl`, keyed by the exact request (model, effort, system prompt and full prompt), so the same request is never sent twice.

In [ ]:
SCHEMA = {"type": "object", "properties": {"level": {"type": "string", "enum": LEVELS}},
          "required": ["level"], "additionalProperties": False}
ANSWERS_FILE = os.path.join(OUTPUT_DIR, f"llm_answers_{DIMENSION}.jsonl")


def request_key(before, after):
    return hashlib.sha1(json.dumps([MODEL, EFFORT, SYSTEM_PROMPT, before, after]).encode()).hexdigest()


answers = {}
if os.path.exists(ANSWERS_FILE):
    with open(ANSWERS_FILE, encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            answers[record["key"]] = record
print(f"{len(answers)} saved answers")


def ask(before, after):
    # One independent request: the level chosen by Claude, or None (refusal or error).
    content = [{"type": "text", "text": before, "cache_control": {"type": "ephemeral"}}] if before else []
    content.append({"type": "text", "text": after})
    kwargs = dict(model=MODEL, max_tokens=MAX_TOKENS, messages=[{"role": "user", "content": content}],
                  output_config={"effort": EFFORT, "format": {"type": "json_schema", "schema": SCHEMA}})
    if SYSTEM_PROMPT:
        kwargs["system"] = SYSTEM_PROMPT
    if USE_FALLBACK:
        kwargs.update(betas=["server-side-fallback-2026-07-01"], fallbacks="default")
    try:
        response = client.beta.messages.create(**kwargs)
    except anthropic.APIStatusError as error:  # not saved, so that it is retried at the next run
        return {"level": None, "stop_reason": f"error {error.status_code}: {error.message}", "save": False}
    level = None
    if response.stop_reason not in ("refusal", "max_tokens"):
        text = next((block.text for block in response.content if block.type == "text"), None)
        level = json.loads(text).get("level") if text else None
    usage = response.usage
    return {"level": level, "stop_reason": response.stop_reason, "served_by": response.model,
            "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
            "cache_read": usage.cache_read_input_tokens or 0, "cache_write": usage.cache_creation_input_tokens or 0,
            "save": True}


def run_requests(jobs):
    # Sends the requests not answered yet. Within a group sharing the same prompt start, the first
    # request is sent alone (it writes the cache), then the others in parallel (they read it).
    todo = {}
    for job in jobs:
        if job["key"] not in answers:
            todo.setdefault(job["before"], {})[job["key"]] = job
    done = 0
    with open(ANSWERS_FILE, "a", encoding="utf-8") as out, ThreadPoolExecutor(MAX_WORKERS) as pool:
        for before, group in todo.items():
            group = list(group.values())
            results = [ask(before, group[0]["after"])]
            results += list(pool.map(lambda job: ask(before, job["after"]), group[1:]))
            for job, result in zip(group, results):
                if result.pop("save"):
                    record = {"key": job["key"], **result}
                    answers[job["key"]] = record
                    out.write(json.dumps(record) + "\n")
                else:
                    print(job["id"], result["stop_reason"])
            out.flush()
            done += len(group)
            print(f"  {done} / {sum(len(g) for g in todo.values())} requests")

## Jobs and cost estimate

In [ ]:
ZERO_SHOT = "none (zero-shot)"


def make_jobs(seed):
    # All the requests of one split: few-shot with each label's examples, and zero-shot (seed-independent).
    folds, _ = load_or_create_split(df.assign(level=df[LABELS[STRATIFY]]), DIMENSION, f"strat{STRATIFY}_seed{seed}",
                                    SPLIT_DIR, M, N_FOLDS, seed)
    jobs = []
    for source, column in LABELS.items():
        examples = load_or_create_examples(folds, df.set_index("id").loc[folds["id"], column],
                                           f"{DIMENSION}_strat{STRATIFY}_{source}_seed{seed}", SPLIT_DIR, M, seed)
        for fold in sorted(folds["fold"].unique()):
            examples_text = format_examples(examples[examples["fold"] == fold], seed * 100 + fold)
            for i in folds.loc[folds["fold"] == fold, "id"]:
                before, after = build_prompt(examples_text, texts[i])
                jobs.append({"mode": f"examples: {source}", "seed": seed, "fold": fold, "id": i,
                             "before": before, "after": after, "key": request_key(before, after)})
    for i in df["id"]:
        before, after = build_prompt(NO_EXAMPLES_TEXT, texts[i])
        jobs.append({"mode": ZERO_SHOT, "seed": seed, "fold": 0, "id": i,
                     "before": before, "after": after, "key": request_key(before, after)})
    return jobs


jobs = [job for seed in range(SEED, SEED + N_REPEATS) for job in make_jobs(seed)]
missing = {job["key"]: job for job in jobs if job["key"] not in answers}
groups = {}
for job in missing.values():
    groups.setdefault(job["before"], []).append(job)

tokens = lambda text: len(text) / 4  # rough: about 4 characters per token
cost = sum(tokens(before) * (PRICES["cache_write"] + (len(group) - 1) * PRICES["cache_read"])
           + sum(tokens(job["after"]) for job in group) * PRICES["input"]
           + len(group) * OUTPUT_TOKENS_GUESS * PRICES["output"] for before, group in groups.items()) / 1e6
print(f"{len(jobs)} classifications, {len(missing)} requests not answered yet, in {len(groups)} prompt groups")
print(f"rough cost of the missing requests with {MODEL}: ${cost:.2f} (cache assumed; output {OUTPUT_TOKENS_GUESS} tokens/request)")

In [ ]:
if not RUN_API:
    raise RuntimeError("Check the estimate above, then set RUN_API = True and rerun to send the requests.")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY (or an `ant auth login` profile)
run_requests(missing.values())

## Predictions

In [ ]:
records = []
for job in jobs:
    answer = answers.get(job["key"], {})
    records.append({"mode": job["mode"], "seed": job["seed"], "fold": job["fold"], "id": job["id"],
                    "pred": answer.get("level"), "stop_reason": answer.get("stop_reason"),
                    "served_by": answer.get("served_by")})
all_predictions = pd.DataFrame(records).merge(df[["id", *LABELS.values()]], on="id")
all_predictions.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_predictions_{DIMENSION}.csv"), index=False)

print("answers that are not a level (refusal, error, missing):",
      all_predictions["pred"].isna().sum(), "of", len(all_predictions))
print(all_predictions.groupby("mode")["served_by"].value_counts(dropna=False).to_string())
usage = pd.DataFrame([a for a in answers.values() if "input_tokens" in a])
if len(usage):
    spent = (usage["input_tokens"] * PRICES["input"] + usage["cache_write"] * PRICES["cache_write"]
             + usage["cache_read"] * PRICES["cache_read"] + usage["output_tokens"] * PRICES["output"]).sum() / 1e6
    print(f"tokens of the saved answers: {usage[['input_tokens', 'cache_write', 'cache_read', 'output_tokens']].sum().to_dict()}"
          f", about ${spent:.2f}")

## Results

Same table as the other notebooks: each mode (rows) scored against each label (columns). Answers that are not a level count as errors in accuracy and macro-F1 and are left out of $\kappa$ (their number is printed above).

In [ ]:
METHOD = "LLM-as-a-judge"


def score(predictions):
    # Scores of every mode against every label, on the predictions of one split.
    results = []
    for mode, pred in predictions.groupby("mode", sort=False):
        for target, column in LABELS.items():
            scored = pred.assign(true=pred[column], pred=pred["pred"].fillna("none"))
            readable = scored[scored["pred"].isin(LEVELS)]
            row = evaluate({(DIMENSION, mode): readable}, method=METHOD, reference=target)
            row = row[row["fold"] == "all"].copy()
            # unreadable answers count as errors in accuracy and macro-F1
            row["accuracy"] = (scored["pred"] == scored["true"]).mean()
            row["macro_f1"] = f1_score(scored["true"], scored["pred"], labels=LEVELS, average="macro")
            row["n"], row["unreadable"] = len(scored), len(scored) - len(readable)
            results.append(row)
    return pd.concat(results, ignore_index=True).rename(columns={"setting": "examples", "reference": "evaluated against"})


main = all_predictions[all_predictions["seed"] == SEED]
results = score(main).drop(columns="fold")
results.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_results_{DIMENSION}.csv"), index=False)
results["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
                             zip(results["kappa"], results["kappa_ci_low"], results["kappa_ci_high"])]
results.pivot(index="examples", columns="evaluated against", values=["accuracy", "macro_f1", "kappa [95% CI]"])

In [ ]:
plot_confusion({(f"vs {target}", mode): pred[pred["pred"].isin(LEVELS)].assign(true=pred[column])
                for target, column in LABELS.items() for mode, pred in main.groupby("mode", sort=False)},
               "true", os.path.join(OUTPUT_DIR, f"human_llm_confusion_{DIMENSION}.png"))
plt.show()

With `N_REPEATS > 1`: mean ± SD over the random splits (zero-shot does not depend on the split).

In [ ]:
if N_REPEATS > 1:
    repeats = pd.concat([score(part).assign(seed=seed) for seed, part in all_predictions.groupby("seed")])
    display(repeats.groupby(["examples", "evaluated against"])[["accuracy", "macro_f1", "kappa"]]
            .agg(["mean", "std"]).round(3))

## Does each LLM respect the requested level?

As in the other notebooks: for the videos of each generator model, the share whose level found by Claude equals the reference level, in each mode, with the human level as a comparison. Accuracy with a 95% bootstrap CI; $\chi^2$ test of equal accuracy across models within each mode.

In [ ]:
from scipy.stats import chi2_contingency


def accuracy_with_ci(correct, seed=0, n_bootstrap=1000):
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct, dtype=float)
    resamples = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(n_bootstrap)]
    return (correct.mean(), *np.percentile(resamples, [2.5, 97.5]))


raw = pd.read_csv(LABELS_FILE)
if GENERATOR_COLUMN is None:
    GENERATOR_COLUMN = next((c for c in raw.columns if re.search("model|generator|llm", c, re.IGNORECASE)), None)
print("generator column:", GENERATOR_COLUMN)

if "reference" not in LABELS or GENERATOR_COLUMN is None:
    print("This section needs a reference level (REFERENCE_COLUMN) and a generator column (GENERATOR_COLUMN).")
else:
    generators = dict(zip([f"row{k}" for k in raw.index], raw[GENERATOR_COLUMN].astype(str)))
    modes = {mode: pred[["id", "pred", "reference"]] for mode, pred in main.groupby("mode", sort=False)}
    modes["humans"] = df.assign(pred=df["level"])[["id", "pred", "reference"]]
    rows, tests = [], []
    for mode, pred in modes.items():
        pred = pred.assign(generator=pred["id"].map(generators), correct=pred["pred"] == pred["reference"])
        for generator, part in [*pred.groupby("generator"), ("all", pred)]:
            accuracy, low, high = accuracy_with_ci(part["correct"])
            rows.append({"mode": mode, "generator": generator, "n": len(part), "accuracy": accuracy,
                         "low": low, "high": high, "accuracy [95% CI]": f"{accuracy:.2f} [{low:.2f}, {high:.2f}]"})
        table = pd.crosstab(pred["generator"], pred["correct"])
        if table.shape[1] == 2 and len(table) > 1:
            chi2_value, p_value, dof, _ = chi2_contingency(table)
            tests.append({"mode": mode, "chi2": chi2_value, "dof": dof, "p (equal accuracy across models)": p_value})
    respect = pd.DataFrame(rows)
    respect.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_respect_by_generator_{DIMENSION}.csv"), index=False)
    display(respect.pivot(index="mode", columns="generator", values="accuracy [95% CI]").reindex(list(modes)))
    display(pd.DataFrame(tests).round(4))

    plot = respect[respect["generator"] != "all"]
    mode_order = ["humans", *[mode for mode in modes if mode != "humans"]]
    generator_order = sorted(plot["generator"].unique())
    width = 0.8 / len(generator_order)
    fig, ax = plt.subplots(figsize=(9.5, 4))
    for k, generator in enumerate(generator_order):
        part = plot[plot["generator"] == generator].set_index("mode").reindex(mode_order)
        x = np.arange(len(mode_order)) + (k - (len(generator_order) - 1) / 2) * width
        ax.bar(x, part["accuracy"], width, label=generator,
               yerr=[part["accuracy"] - part["low"], part["high"] - part["accuracy"]], capsize=3)
    ax.axvline(0.5, c="black", lw=0.8)
    ax.axhline(1 / len(LEVELS), ls="--", c="grey", lw=1, label="chance")
    ax.set_xticks(np.arange(len(mode_order)), mode_order)
    ax.set_ylabel("level found = requested level")
    ax.set_title(f"{METHOD}: does each LLM respect the requested {DIMENSION} level?")
    ax.legend(title="generator", fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"human_llm_respect_by_generator_{DIMENSION}.png"), dpi=300)
    plt.show()

## Comparison with FGW and embeddings

If `human_fgw.ipynb` and `human_embeddings.ipynb` were run with the same parameters (same splits).

In [ ]:
others = []
for prefix in ["human_fgw", "human_embedding"]:
    for kind in ["results", "unsupervised_results"]:
        path = os.path.join(OUTPUT_DIR, f"{prefix}_{kind}_{DIMENSION}.csv")
        if os.path.exists(path):
            other = pd.read_csv(path)
            others.append(other[other["fold"].astype(str) == "all"] if "fold" in other else other)
if others:
    both = pd.concat([results, *others], ignore_index=True)
    display(both.pivot_table(index=["examples", "evaluated against"], columns="method",
                             values=["accuracy", "kappa"]).round(3))